# 04 — Classical Optimization

OR-Tools CVRP solver against real CVRPLIB ground truth and a real Amazon Last-Mile route — the baseline every Phase 3 quantum routing result is measured against. All results are **LIVE**.

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np

from qsci_core.data import loaders
from qsci_core.optimization.classical.or_tools_vrp import solve_cvrp
from qsci_core import mlflow_utils

mlflow_utils.set_tracking_uri()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
print("project root:", PROJECT_ROOT)


project root: D:\Projects\QML project


## CVRPLIB — OR-Tools vs. published best-known cost

Status: **LIVE**. Each instance's best-known cost comes from its real `.sol` file, not a hardcoded number.

In [2]:
CVRPLIB_INSTANCES = ["A-n32-k5", "A-n33-k5", "A-n37-k5", "B-n31-k5"]

cvrplib_results = []
for name in CVRPLIB_INSTANCES:
    instance, solution = loaders.load_cvrplib_instance(name)
    n_vehicles = int(name.rsplit("-k", 1)[1])  # instance name convention: "...-k<n_vehicles>"

    coords = np.asarray(instance["node_coord"], dtype=float)
    demands = np.asarray(instance["demand"], dtype=float)
    capacity = instance["capacity"]

    result = solve_cvrp(coords, demands, capacity, n_vehicles, depot=0, time_limit_seconds=15)
    best_known = solution["cost"]
    gap_pct = (result.total_distance - best_known) / best_known * 100

    cvrplib_results.append({
        "instance": name, "status": result.status, "or_tools_cost": result.total_distance,
        "best_known_cost": best_known, "gap_pct": gap_pct, "vehicles_used": result.vehicles_used,
    })
    print(f"{name}: OR-Tools={result.total_distance:.1f}  best-known={best_known}  gap={gap_pct:+.2f}%  vehicles={result.vehicles_used}/{n_vehicles}")

max_gap = max(r["gap_pct"] for r in cvrplib_results)
print(f"\nmax gap across all {len(cvrplib_results)} instances: {max_gap:.2f}%")


A-n32-k5: OR-Tools=787.1  best-known=784  gap=+0.39%  vehicles=5/5


A-n33-k5: OR-Tools=662.1  best-known=661  gap=+0.17%  vehicles=5/5


A-n37-k5: OR-Tools=672.5  best-known=669  gap=+0.52%  vehicles=5/5


B-n31-k5: OR-Tools=676.1  best-known=672  gap=+0.61%  vehicles=5/5

max gap across all 4 instances: 0.61%


## Amazon Last-Mile — OR-Tools vs. the actual historical driver

Status: **LIVE**. Real route (smallest by stop count in the training set, 33 stops incl. depot), real travel-time matrix (`travel_times.json`, seconds), real historical driven order (`actual_sequences.json`). This is an open route — starts at the station, visits every stop once, does not return — modeled via a zero-cost dummy end node so OR-Tools' arc-cost objective matches what the real driver was actually doing (not a round trip).

Caveat: a real driver optimizes more than pure travel time — delivery windows, package load order, traffic knowledge, personal routine. A gap here isn't a claim that routing software beats human drivers in general.

In [3]:
last_mile = loaders.load_last_mile(RAW_DIR / "last_mile")
routes, packages_unused = last_mile["route_data"], last_mile.get("package_data")
actual_sequences = last_mile["actual_sequences"]
travel_times = last_mile["travel_times"]

# smallest real route by stop count -- keeps OR-Tools tractable while staying real data, not a toy instance
route_sizes = sorted(((rid, len(r["stops"])) for rid, r in routes.items()), key=lambda kv: kv[1])
route_id, n_stops = route_sizes[0]
stops = routes[route_id]["stops"]
station_id = next(sid for sid, s in stops.items() if s["type"] == "Station")
stop_ids = [station_id] + [sid for sid in stops if sid != station_id]
n = len(stop_ids)
idx = {sid: i for i, sid in enumerate(stop_ids)}

print(f"route: {route_id}  |  station: {routes[route_id]['station_code']}  |  date: {routes[route_id]['date_YYYY_MM_DD']}  |  stops incl. depot: {n}")

real_matrix = np.zeros((n, n))
for a in stop_ids:
    for b in stop_ids:
        real_matrix[idx[a], idx[b]] = travel_times[route_id][a][b]

actual_order = [sid for sid, _ in sorted(actual_sequences[route_id]["actual"].items(), key=lambda kv: kv[1])]
actual_travel_seconds = sum(travel_times[route_id][actual_order[i]][actual_order[i + 1]] for i in range(len(actual_order) - 1))
print(f"actual driver total travel time: {actual_travel_seconds:.1f}s ({actual_travel_seconds/60:.1f} min)")


route: RouteID_64cb7ba5-342d-46db-9e04-962248c6f667  |  station: DCH1  |  date: 2018-07-19  |  stops incl. depot: 33
actual driver total travel time: 3660.9s (61.0 min)


In [4]:
# dummy end node with zero-cost edges everywhere -- turns the closed-tour
# CVRP solver into an open-route solver (start at depot, visit all stops, no return leg)
n_open = n + 1
open_matrix = np.zeros((n_open, n_open))
open_matrix[:n, :n] = real_matrix
demands = np.ones(n_open)
demands[idx[station_id]] = 0  # depot
demands[n] = 0  # dummy end node

result = solve_cvrp(
    coords=np.zeros((n_open, 2)),  # unused: distance_matrix overrides Euclidean
    demands=demands, capacity=n_open, n_vehicles=1, depot=idx[station_id],
    distance_matrix=open_matrix, time_limit_seconds=20,
)
or_tools_route = [stop_ids[node] for node in result.routes[0] if node != n]  # strip dummy node, map back to real stop ids
print("status:", result.status)
print("OR-Tools total travel time:", f"{result.total_distance:.1f}s ({result.total_distance/60:.1f} min)")
print("stops covered:", len(or_tools_route), "/", n - 1)

improvement_pct = (actual_travel_seconds - result.total_distance) / actual_travel_seconds * 100
print(f"\nOR-Tools vs. actual driver: {improvement_pct:+.1f}% travel-time difference (positive = OR-Tools route is faster)")


status: OPTIMAL_OR_FEASIBLE
OR-Tools total travel time: 2561.2s (42.7 min)
stops covered: 32 / 32

OR-Tools vs. actual driver: +30.0% travel-time difference (positive = OR-Tools route is faster)


## mlflow tracking + honest summary

Status: **LIVE**.

In [5]:
import mlflow

with mlflow_utils.tracked_run(
    "classical_routing", "or_tools_cvrplib", dataset_version="cvrplib-augerat-A-B",
    seed=0, params={"instances": ",".join(CVRPLIB_INSTANCES), "time_limit_seconds": 15},
) as log:
    log({"max_gap_pct": max_gap, "mean_gap_pct": sum(r["gap_pct"] for r in cvrplib_results) / len(cvrplib_results)})

with mlflow_utils.tracked_run(
    "classical_routing", f"or_tools_lastmile_{route_id}", dataset_version="amazon-last-mile-almrrc2021",
    seed=0, params={"route_id": route_id, "n_stops": n - 1, "time_limit_seconds": 20},
) as log:
    log({"or_tools_seconds": result.total_distance, "actual_driver_seconds": actual_travel_seconds, "improvement_pct": improvement_pct})

runs = mlflow.search_runs(experiment_names=["classical_routing"])
print(runs[["tags.mlflow.runName", "tags.git_commit", "metrics.runtime_seconds"]])

print("\n=== Honest summary ===")
print(f"CVRPLIB: OR-Tools solved {len(cvrplib_results)}/{len(cvrplib_results)} instances feasibly; gap vs. published best-known ranged up to {max_gap:.2f}%.")
print(f"Last-Mile: OR-Tools' travel-time-optimized route was {improvement_pct:+.1f}% vs. the actual historical driver on route {route_id} ({n-1} stops) — a pure travel-time comparison, not a claim about overall route quality.")


2026/08/18 21:35:58 INFO mlflow.tracking.fluent: Experiment with name 'classical_routing' does not exist. Creating a new experiment.


                                 tags.mlflow.runName  \
0  or_tools_lastmile_RouteID_64cb7ba5-342d-46db-9...   
1                                   or_tools_cvrplib   

                            tags.git_commit  metrics.runtime_seconds  
0  0fbeac6a10fffdad02892249ff0f0fca06636dfd                  0.09780  
1  0fbeac6a10fffdad02892249ff0f0fca06636dfd                  0.41455  

=== Honest summary ===
CVRPLIB: OR-Tools solved 4/4 instances feasibly; gap vs. published best-known ranged up to 0.61%.
Last-Mile: OR-Tools' travel-time-optimized route was +30.0% vs. the actual historical driver on route RouteID_64cb7ba5-342d-46db-9e04-962248c6f667 (32 stops) — a pure travel-time comparison, not a claim about overall route quality.
